# 05 — Exploratory analysis of flagged matches

**Status: EXPLORATORY / post-hoc — hypothesis-generating, not confirmatory.**
This notebook mines the flagged set *after* the fact. Nothing here was pre-registered,
so findings do **not** carry the evidentiary weight of `04_hypothesis_tests` and must
never be presented alongside the pre-registered results as if they were equivalent.

Reads the frozen artifact `data/processed/scored_matches.parquet` (+ `features.parquet`
for odds columns). Does **not** re-run or depend on the models in `03` — fully decoupled.

### Two hard rails
1. **No ground truth.** A flag = "unusual odds behaviour relative to baseline," nothing
   more. Nothing here supports "manipulated" — only descriptive characterisation.
2. **Every interesting pattern has a boring explanation** (efficient markets, structural
   league differences, the 25/26 coverage gap) that must be ruled out *before* it counts
   as interesting.

**Excluded by design:** no named-team frequency tables. With no ground truth, a leaderboard
of real clubs under an "anomalies" header is methodologically void and a reputational/legal
liability. Team-level structure, if ever needed, stays aggregate and unnamed.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

def find_repo_root(marker="data/processed"):
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / marker).is_dir():
            return cand
    raise FileNotFoundError(f"couldn't locate '{marker}' from {Path.cwd()} upward")

PROC = find_repo_root() / "data" / "processed"

scored = pd.read_parquet(PROC / "scored_matches.parquet")
feat   = pd.read_parquet(PROC / "features.parquet")

# guard the join key is unique in BOTH frames before any merge (validate= relies on this)
key = ["match_date", "home_team", "away_team"]
for name, frame in [("scored", scored), ("features", feat)]:
    dups = int(frame.duplicated(key).sum())
    assert dups == 0, f"{name} has {dups} duplicate keys on {key} — join will fanout"
print(f"loaded scored {scored.shape}, features {feat.shape} | key unique in both ✓")

loaded scored (8915, 18), features (8915, 114) | key unique in both ✓


## 1 · Outcome character, line-movement, and clustering

Three descriptive slices of the universal-model flags:

- **A. Outcome mix** — do flagged games resolve to H/D/A differently than baseline?
  *Boring null:* spread/drift are flagging features and high-spread games are structurally
  more often upsets, so an odd outcome mix may just be "extreme-odds games behave like
  extreme-odds games."
- **B. Line movement toward the result** — did closing odds shorten toward the eventual
  winner *more* on flagged games? The only integrity-flavoured angle. *Circularity trap:*
  drift drives the flags, so flagged games have large drift **by construction** — large
  drift isn't interesting, *directional* drift that predicts the winner is. Interesting only
  if the flagged row is meaningfully more negative than the not-flagged row. **Pinnacle is
  the column that matters** (sharp money); B365 may just follow it. If Pinnacle drift-toward-
  winner is *not* stronger on flagged games, that's the efficient-markets null — a fine,
  unsensational result.
- **C. Season clustering** — *watch:* `tier_only` will likely spike in 25/26 because
  `pinnacle_missing` is a flagging feature and that's the coverage-gap season. Read the
  `both` column (models agree) as the robust signal, not `tier_only`.

In [2]:
key = ["match_date", "home_team", "away_team"]
df = scored.merge(
    feat[key + ["b365_drift_h", "b365_drift_d", "b365_drift_a",
                "pinnacle_drift_h", "pinnacle_drift_d", "pinnacle_drift_a"]],
    on=key, how="left", validate="one_to_one")   # validate guards against row fanout

res = df["full_time_result"]
# drift OF THE REALISED OUTCOME (drift = close − open; negative = market shortened toward it)
for bk in ["b365", "pinnacle"]:
    df[f"{bk}_drift_winner"] = np.select(
        [res == "H", res == "D", res == "A"],
        [df[f"{bk}_drift_h"], df[f"{bk}_drift_d"], df[f"{bk}_drift_a"]])

print("=== A. outcome character: flagged vs not (universal) ===")
print(df.groupby("if_u_flag")["full_time_result"].value_counts(normalize=True).unstack().round(3))

print("\n=== B. did the line move TOWARD the winner? (mean signed drift of realised outcome) ===")
print("negative = shortened toward eventual result; INTERESTING only if flagged << not-flagged")
print(df.groupby("if_u_flag")[["b365_drift_winner", "pinnacle_drift_winner"]].mean().round(4))

print("\n=== C. clustering: flag_agreement by season ===")
print(pd.crosstab(df["season"], df["flag_agreement"]))

=== A. outcome character: flagged vs not (universal) ===
full_time_result      A      D      H
if_u_flag                            
False             0.316  0.259  0.425
True              0.269  0.121  0.610

=== B. did the line move TOWARD the winner? (mean signed drift of realised outcome) ===
negative = shortened toward eventual result; INTERESTING only if flagged << not-flagged
           b365_drift_winner  pinnacle_drift_winner
if_u_flag                                          
False                -0.0097                 0.0016
True                  0.0091                 0.0340

=== C. clustering: flag_agreement by season ===
flag_agreement  both  neither  tier_only  universal_only
season                                                  
1920              54     1137          6              35
2021              49     1270         23               3
2122              52     1231         14               9
2223              42     1152         25              18
2324           

In [3]:
clean = df[~df["pinnacle_missing"].astype(bool)]
print("B (Pinnacle-present rows only):")
print(clean.groupby("if_u_flag")[["b365_drift_winner", "pinnacle_drift_winner"]].mean().round(4))
print("\nflagged n:", int(clean["if_u_flag"].sum()), "of", len(clean))

B (Pinnacle-present rows only):
           b365_drift_winner  pinnacle_drift_winner
if_u_flag                                          
False                -0.0096                 0.0016
True                  0.0133                 0.0340

flagged n: 414 of 8187


## 2 · Greece-specific cut (exploratory)

The §1 results pooled all flagged games, but the set is dominated by Greece (most-flagged
league) blended with three near-baseline leagues. This cut asks whether Greece's flagged
games behave **like the rest** (→ the "non-informative drift" finding is league-general) or
**differently** (→ Greece's anomalies are qualitatively distinct, deepening the headline
outlier story). Requires `df` from §1.

In [4]:
g1 = df["league"] == "G1"
clean = ~df["pinnacle_missing"].astype(bool)   # Pinnacle-present rows only

print("=== A. Greece — outcome mix: flagged vs not ===")
print(df[g1].groupby("if_u_flag")["full_time_result"]
        .value_counts(normalize=True).unstack().round(3))

print("\n=== B. drift-toward-winner among FLAGGED games: Greece vs rest (Pinnacle-present) ===")
fl = df[df["if_u_flag"] & clean].copy()
fl["is_greece"] = fl["league"].eq("G1")
print(fl.groupby("is_greece")[["b365_drift_winner", "pinnacle_drift_winner"]].mean().round(4))
print("n — Greece:", int((fl["league"] == "G1").sum()), "| rest:", int((fl["league"] != "G1").sum()))

print("\n=== B2. within Greece: flagged vs not (Pinnacle-present) ===")
print(df[g1 & clean].groupby("if_u_flag")[["b365_drift_winner", "pinnacle_drift_winner"]].mean().round(4))

print("\n=== C. Greece flagged — dominant family (convenience label) ===")
print(df.loc[g1 & df["if_u_flag"], "dom_family_u"].value_counts())

=== A. Greece — outcome mix: flagged vs not ===
full_time_result      A      D      H
if_u_flag                            
False             0.320  0.287  0.393
True              0.239  0.142  0.619

=== B. drift-toward-winner among FLAGGED games: Greece vs rest (Pinnacle-present) ===
           b365_drift_winner  pinnacle_drift_winner
is_greece                                          
False                -0.0194                 0.0273
True                  0.0686                 0.0453
n — Greece: 154 | rest: 260

=== B2. within Greece: flagged vs not (Pinnacle-present) ===
           b365_drift_winner  pinnacle_drift_winner
if_u_flag                                          
False                -0.0049                 0.0085
True                  0.0686                 0.0453

=== C. Greece flagged — dominant family (convenience label) ===
dom_family_u
spread      170
reversal      3
drift         3
Name: count, dtype: int64


## 3 · Favourite-confound check (does the Greece drift effect survive?)

§2's away-from-winner drift in Greek flagged games could be mechanical: short-priced
favourites have little room to shorten but lots to drift out, and upset (longshot) winners
produce large positive `drift_winner` by construction. If the effect lives only in upsets,
it's the confound; if it holds **within favourite-won games**, it's a real market-quality
signal. Split Greek flagged games by `favourite_won`; report mean **and** median (a few
large-drift upsets can carry a mean) with n per cell.

In [5]:
odds = ["avg_close_h", "avg_close_d", "avg_close_a"]   # market consensus closing odds
g = df.merge(feat[key + odds], on=key, how="left", validate="one_to_one")

# favourite = lowest closing odds; favourite_won = that outcome matches the result
fav = np.array(["H", "D", "A"])[g[odds].values.argmin(axis=1)]
g["favourite_won"] = fav == g["full_time_result"].values

gr = g[(g["league"] == "G1") & g["if_u_flag"] & ~g["pinnacle_missing"].astype(bool)]
print(f"Greek flagged (Pinnacle-present): n={len(gr)} | favourite-won share={gr['favourite_won'].mean():.3f}\n")
for col in ["b365_drift_winner", "pinnacle_drift_winner"]:
    print(f"{col}:")
    print(gr.groupby("favourite_won")[col].agg(n="size", mean="mean", median="median").round(4), "\n")

Greek flagged (Pinnacle-present): n=154 | favourite-won share=0.851

b365_drift_winner:
                 n    mean  median
favourite_won                     
False           23  0.5152     0.1
True           131 -0.0098     0.0 

pinnacle_drift_winner:
                 n   mean  median
favourite_won                    
False           23  0.337    0.14
True           131 -0.006   -0.01 



In [6]:
pool = g[g["if_u_flag"] & ~g["pinnacle_missing"].astype(bool)]   # all leagues
print(f"ALL flagged (Pinnacle-present): n={len(pool)} | favourite-won share={pool['favourite_won'].mean():.3f}\n")
for col in ["b365_drift_winner", "pinnacle_drift_winner"]:
    print(f"{col}:")
    print(pool.groupby("favourite_won")[col].agg(n="size", mean="mean", median="median").round(4), "\n")

ALL flagged (Pinnacle-present): n=414 | favourite-won share=0.836

b365_drift_winner:
                 n    mean  median
favourite_won                     
False           68  0.1243     0.0
True           346 -0.0085     0.0 

pinnacle_drift_winner:
                 n    mean  median
favourite_won                     
False           68  0.2484   0.015
True           346 -0.0081  -0.010 



## Summary — exploratory analysis of flagged matches

**Status: EXPLORATORY / post-hoc.** Hypothesis-generating, not confirmatory. None of this
was pre-registered; it does not carry the weight of `04_hypothesis_tests` and must never be
presented alongside the pre-registered results as equivalent. Reads the frozen artifact
`scored_matches.parquet`; does not re-run the `03` models.

### Question
What characterises the matches the universal Isolation Forest flags (446 of 8,915, ~5%)?

### What we found, and what survived scrutiny

**1. Flagged games are lopsided, favourite-heavy fixtures.** Across the flagged set: ~84%
favourite-won, home-win-skewed (61% vs 42.5% baseline), draws halved (≈12% vs 26%). This is
the screen flagging one-sided markets — consistent with spread being its strongest feature
(SHAP, `03`). **Survives** as a descriptive fact. Most concentrated in Greece (85%
favourite-won).

**2. An apparent "odds drift away from the winner" — REFUTED as a real signal.** Initial cuts
(pooled and Greece-specific) showed the winning outcome's odds drifting *out* before kickoff
(pooled Pinnacle +0.034; Greece +0.045), which would be the *opposite* of a manipulation
signature (informed money moves *toward* a known result, not away). **A pre-stated
favourite-confound check killed it:** splitting flagged games by `favourite_won`, the
favourite-won majority (84%) shows **zero** drift (mean and median ≈ 0 on both books); the
entire positive figure comes from a small upset minority (pooled n=68, Greece n=23), where a
longshot winning means the market had drifted away from it *by construction*. The split
reconstructs the original pooled figures to four decimals (b365 +0.0133, Pinnacle +0.0340),
proving the "signal" was a **sample-composition artifact**, not informative movement.

**3. No temporal clustering.** Model-agreed flags (`both`) are flat (~42–54/season) across all
seven seasons; the `tier_only` spike in 25/26 is the `pinnacle_missing` coverage artifact,
not a real pattern.

### Conclusion
The anomaly screen flags unusual market **shape** (lopsided, favourite-heavy markets), not
informative or suspicious market **movement**. After controlling for fixture composition,
flagged games' pre-kickoff odds carry no directional information — no drift toward the true
result (no manipulation signature) and none away from it (no market-quality signal). This
**reinforces** the market-efficiency framing: the screen does its job, and disciplined
follow-through shows the exciting-looking pattern was mechanical.